In [22]:
import pandas as pd
import numpy as np

movies = pd.read_csv('movies.csv')
ratings = pd.read_csv('ratings.csv')

print("--- MOVIES ---")
print(movies.shape)
print(movies.head())

print("\n--- RATINGS ---")
print(ratings.shape)
print(ratings.head())

--- MOVIES ---
(9742, 3)
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  

--- RATINGS ---
(100836, 4)
   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931


In [23]:
print(f"Number of unique users: {ratings['userId'].nunique()}")
print(f"Number of unique movies:  {ratings['movieId'].nunique()}")
print(f"Number of ratings:        {len(ratings)}")

# What does the genres column actually look like?
print("\nSample genres:")
print(movies['genres'].head(10))

Number of unique users: 610
Number of unique movies:  9724
Number of ratings:        100836

Sample genres:
0    Adventure|Animation|Children|Comedy|Fantasy
1                     Adventure|Children|Fantasy
2                                 Comedy|Romance
3                           Comedy|Drama|Romance
4                                         Comedy
5                          Action|Crime|Thriller
6                                 Comedy|Romance
7                             Adventure|Children
8                                         Action
9                      Action|Adventure|Thriller
Name: genres, dtype: str


In [24]:
# str.get_dummies(sep='|') - takes a text column, splits each row on the
# separator '|', then makes one new column per unique genre it finds.
# Each cell becomes 1 (movie has that genre) or 0 (it doesn't).
genre_matrix = movies['genres'].str.get_dummies(sep='|')


# .shape - returns (number_of_rows, number_of_columns) as a tuple
print(genre_matrix.shape)


# .head() - first 5 rows, so we can eyeball the 0/1 layout
genre_matrix.head()

(9742, 20)


,(no genres listed),Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,0,0,1,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0
1,0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0
3,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0
4,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [25]:
# How many movies have NO genres listed?

no_genre_count = genre_matrix['(no genres listed)'].sum()
# .sum() on a 0/1 column = counts the 1s, since 1+1+1... = total number of movies flagged

print(f"Movies with no genres: {no_genre_count}")
print(f"That's {no_genre_count / len(genre_matrix) * 100:.2f}% of all movies")
# :.2f - format the number to 2 decimal places, so we get e.g. 0.35% not 0.3491...%

Movies with no genres: 34
That's 0.35% of all movies


In [26]:
# .drop() - removes a row or column from a DataFrame.
# columns='(no genres listed)' - names the column to remove
#   (the exact string, parentheses and all, must match the header).
# We reassign to genre_matrix so the change sticks.
genre_matrix = genre_matrix.drop(columns='(no genres listed)')

# Confirm the column is gone - shape's column count should drop by 1.
print(genre_matrix.shape)
genre_matrix.head()

(9742, 19)


,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,0,1,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0
1,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0
3,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0
4,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [27]:
# Import the function. sklearn groups similarity/distance tools under
# "pairwise" - meaning "compute a value for every PAIR of rows."
from sklearn.metrics.pairwise import cosine_similarity

# cosine_similarity(genre_matrix) - takes our grid of 9742 movies x 19 genres
# and compares EVERY movie against EVERY other movie. For each pair it returns
# a number from 0 (no shared genres, 90° apart) to 1 (identical genres, 0° apart).
similarity = cosine_similarity(genre_matrix)

# similarity is a plain NumPy array (not a DataFrame - no row/column labels).
# .shape tells us how big the result grid is.
print(similarity.shape)

# Peek at the top-left corner: first 5 movies compared against first 5 movies.
print(similarity[:5, :5])

(9742, 9742)
[[1.         0.77459667 0.31622777 0.25819889 0.4472136 ]
 [0.77459667 1.         0.         0.         0.        ]
 [0.31622777 0.         1.         0.81649658 0.70710678]
 [0.25819889 0.         0.81649658 1.         0.57735027]
 [0.4472136  0.         0.70710678 0.57735027 1.        ]]


In [28]:
# We want to look up a movie's row number FROM its title.
# pd.Series(data, index=...) builds a lookup table:
#   - the VALUES are the row numbers (0, 1, 2, ... 9741)
#   - the INDEX (the "keys" we look up by) is the movie titles
# movies.index gives us 0..9741; movies['title'] gives the titles in the same order.
indices = pd.Series(movies.index, index=movies['title'])

# Now indices['some title'] returns that movie's row number.
# Test it on a title we know is row 0:
print(indices['Toy Story (1995)'])

# Show the first few entries so you can see the title -> number mapping:
indices.head()

0


title
Toy Story (1995)                      0
Jumanji (1995)                        1
Grumpier Old Men (1995)               2
Waiting to Exhale (1995)              3
Father of the Bride Part II (1995)    4
dtype: int64

In [29]:
import re   # re - Python's regex (pattern-matching) library

def fix_title(title):
    # turn "Emperor's New Groove, The (2000)" into "The Emperor's New Groove (2000)"

    # pattern explained piece by piece:
    # ^(.+)        - group 1: capture everything from the start (the main title)
    # ,\s          - a literal comma followed by a space (the ", " before the article)
    # (The|A|An)   - group 2: capture exactly one of these three articles
    # \s           - the space before the year
    # (\(\d{4}\))  - group 3: capture the year in brackets, e.g. (2000)
    #                \( and \) are literal brackets; \d{4} is exactly 4 digits
    match = re.match(r'^(.+), (The|A|An) (\(\d{4}\))$', title)

    if match:
        # rebuild as: article + main title + year
        return f"{match.group(2)} {match.group(1)} {match.group(3)}"
        # group(2)=article, group(1)=main title, group(3)=year
    else:
        return title   # no article at the end → leave it untouched

# test it on a few cases
print(fix_title("Emperor's New Groove, The (2000)"))
print(fix_title('Matrix, The (1999)'))
print(fix_title('Toy Story (1995)'))   # no article - should come back unchanged

The Emperor's New Groove (2000)
The Matrix (1999)
Toy Story (1995)


In [30]:
def recommend(title, n=5):
    # n=5 - default number of recommendations; caller can override.

     # STEP 1: title -> row number, using the bridge we just built.
    idx = indices[title]

    # STEP 2: pull this movie's row from the similarity grid.
    # similarity[idx] is a 1D array of 9742 scores: this movie vs every movie.
    # enumerate(...) pairs each score with its position -> (0, score0), (1, score1)...
    # We need those positions so we can map scores back to movies later.

    sim_scores = list(enumerate(similarity[idx]))

    # STEP 3: sort the (position, score) pairs by score, highest first.
    # key=lambda x: x[1] - sort by the SECOND item in each pair (the score),
    #   not the position. reverse=True - biggest scores at the top.

    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # STEP 4: take the top matches, but SKIP the first one.
    # [1:n+1] - position 0 is the movie compared to itself (score 1.0),
    #   which we don't want to recommend. So we start at 1 and take n.

    sim_scores = sim_scores[1:n+1]

    # STEP 5: pull just the positions (row numbers) out of those pairs.

    movies_indices = [i[0] for i in sim_scores]

    # STEP 6: translate row numbers back into titles and return them.
    # .iloc[...] selects rows by integer position from the title column.

    results = movies['title'].iloc[movies_indices]

    return results.apply(fix_title)

recommend('Toy Story (1995)')

1706                                      Antz (1998)
2355                               Toy Story 2 (1999)
2809    The Adventures of Rocky and Bullwinkle (2000)
3000                  The Emperor's New Groove (2000)
3568                            Monsters, Inc. (2001)
Name: title, dtype: str

In [31]:
recommend('Heat (1995)')   # Action|Crime|Thriller - very different from Toy Story


22                      Assassins (1995)
138    Die Hard: With a Vengeance (1995)
156                       The Net (1995)
249          Natural Born Killers (1994)
417                Judgment Night (1993)
Name: title, dtype: str

In [32]:
recommend('Pulp Fiction (1994)')
# Pulp Fiction is Comedy/Crime/Drama/Thriller - a common combination.
# Guess: do you expect the results to feel as "right" as the Toy Story ones did?

520                                          Fargo (1996)
791                                        Freeway (1996)
2453    Man Bites Dog (C'est arrivé près de chez vous)...
3155                           Beautiful Creatures (2000)
4169               Confessions of a Dangerous Mind (2002)
Name: title, dtype: str

In [42]:
# Day 3, Step 1 - get reacquainted with the ratings data

print(ratings.shape)        # (rows, columns) - how many ratings total?
print()
ratings.head()              # peek at the first 5 rows
print()
print(ratings.nunique())    # nunique - counts DISTINCT values in each column
# this tells us: how many unique users? how many unique movies got rated?

(100836, 4)


userId         610
movieId       9724
rating          10
timestamp    85043
dtype: int64


In [44]:
# Day 3, Step 2 - pivot the long ratings list into a wide user-item matrix

user_item = ratings.pivot_table(index='userId', columns='movieId', values='rating')
# pivot_table - reshapes long data into a grid
# index='userId'    - each unique user becomes a ROW
# columns='movieId' - each unique movie becomes a COLUMN
# values='rating'   - the rating fills the cell where a user meets a movie

print(user_item.shape)
user_item.iloc[:5, :5]

(610, 9724)


movieId,1,2,3,4,5
userId,,,,,
1,4.0,NaN,4.0,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN
5,4.0,NaN,NaN,NaN,NaN


In [46]:
# Day 3, Step 3 - measure exactly how sparse the matrix is

total_cells = user_item.shape[0] * user_item.shape[1]
# rows × columns = total possible user-movie combinations

filled_cells = user_item.notna().sum().sum()
# notna()      - True where there's a rating, False where NaN
# .sum().sum() - first sum counts per column, second sums those into one grand total
#                (we chain two sums because the first gives a number PER column)

sparsity = (1 - filled_cells / total_cells) * 100
# fraction empty, turned into a percentage

print(f"Total cells:  {total_cells:,}")       # :, - adds thousands commas, e.g. 5,931,640
print(f"Filled cells: {filled_cells:,}")
print(f"Sparsity:     {sparsity:.2f}% empty")

Total cells:  5,931,640
Filled cells: 100,836
Sparsity:     98.30% empty


In [47]:
# Day 3 Path A, Step 1 - fill the holes and flip the matrix to movies-as-rows
item_user = user_item.fillna(0).T
# .fillna(0) - replace every NaN with 0 (an "unrated" cell becomes a 0)
# .T         - transpose: flip rows and columns, so now MOVIES are rows, USERS are columns
#              we want movies as rows because we're comparing movies to each other

print(item_user.shape)
item_user.iloc[:3, :8]

(9724, 610)


userId,1,2,3,4,5,6,7,8
movieId,,,,,,,,
1,4.0,0.0,0.0,0.0,4.0,0.0,4.5,0.0
2,0.0,0.0,0.0,0.0,0.0,4.0,0.0,4.0
3,4.0,0.0,0.0,0.0,0.0,5.0,0.0,0.0


In [48]:
# Day 3 Path A, Step 2 - compute movie-to-movie similarity from rating patterns
from sklearn.metrics.pairwise import cosine_similarity

item_similarity = cosine_similarity(item_user)
# same function as Day 2 but each movie is now a vector of 610 user-ratings
# instead of a vector of 20 genre-flags
# two movies score high if the SAME USERS rated them similarly

print(item_similarity.shape)

(9724, 9724)


In [49]:
# Day 3 Path A, Step 3 - build lookups that match THIS grid's row order

# the rows of item_similarity are in the same order as item_user's rows,
# which are the movieIds in item_user.index
movie_ids = item_user.index
# item_user.index - the movieIds, in the exact order they appear as rows in the grid

# lookup 1: movieId → its row position in the similarity grid
movieid_to_pos = pd.Series(range(len(movie_ids)), index=movie_ids)
# range(len(...)) - the positions 0,1,2,... become the VALUES
# index=movie_ids - the movieIds become the LABELS we look up BY

# lookup 2: title → movieId, so a human can search by name
title_to_movieid = pd.Series(movies['movieId'].values, index=movies['title'])
# values = movieIds, labels = titles → feed a title, get its movieId

# test the chain on Toy Story (movieId 1)
test_title = 'Toy Story (1995)'
mid = title_to_movieid[test_title]      # title → movieId
pos = movieid_to_pos[mid]               # movieId → grid position
print(f"'{test_title}' → movieId {mid} → grid position {pos}")

'Toy Story (1995)' → movieId 1 → grid position 0


In [50]:
# Day 3 Path A, Step 4 - the collaborative-filtering recommender

def recommend_cf(title, n=5):
    # CF = collaborative filtering

    pos = movieid_to_pos[title_to_movieid[title]]
    # title → movieId → grid position, the two-hop chain we just tested

    sim_scores = list(enumerate(item_similarity[pos]))
    # this movie's similarity to all 9724 movies, paired with grid positions
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:n+1]
    # drop position 0 (the movie itself, score 1.0), take the next n

    rec_positions = [i[0] for i in sim_scores]
    # the grid positions of the recommended movies

    rec_movieids = movie_ids[rec_positions]
    # positions → movieIds, using the same movie_ids index the grid was built from

    # now translate movieIds back to titles via the movies table
    rec_titles = movies[movies['movieId'].isin(rec_movieids)]['title']
    # isin() - keep rows whose movieId is in our recommended list
    
    return rec_titles.apply(fix_title)
    # reuse the Day 2 title-beautifier on the way out

# the moment of truth - compare against Day 2's genre-based results
recommend_cf('Toy Story (1995)')

224     Star Wars: Episode IV - A New Hope (1977)
314                           Forrest Gump (1994)
418                          Jurassic Park (1993)
615          Independence Day (a.k.a. ID4) (1996)
2355                           Toy Story 2 (1999)
Name: title, dtype: str

In [51]:
# Day 3 Part B, Step 1 - install the surprise library
# surprise - a scikit-learn-style library built specifically for recommenders
# run this in a notebook cell; the ! sends the command to the terminal

!pip install scikit-surprise

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 2.7 MB/s eta 0:00:00 MB/s eta 0:00:01:010m


In [53]:
# Day 3 Part B, Step 2 - load the ratings into surprise's format

from surprise import Dataset, Reader

reader = Reader(rating_scale=(0.5, 5.0))
# Reader - tells surprise how to interpret the ratings
# rating_scale=(0.5, 5.0) - the min and max possible rating in MovieLens
#                           (we confirmed earlier ratings run 0.5 to 5.0 in half-steps)

data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)
# load_from_df - builds a surprise Dataset from a pandas DataFrame
# IMPORTANT: surprise requires EXACTLY these three columns, IN THIS ORDER:
#            user, item, rating - nothing else (no timestamp)
# [['userId','movieId','rating']] - double brackets select those 3 columns as a DataFrame

print("Data loaded into surprise format")
print(f"Number of ratings: {data.df.shape[0]:,}")
# data.df - the DataFrame surprise stored internally; check the count matches

Data loaded into surprise format
Number of ratings: 100,836


In [54]:
# Day 3 Part B, Step 3 - split into train/test so we can measure honestly

from surprise.model_selection import train_test_split

trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

# test_size=0.2   - hold back 20% of ratings as a test set the model never trains on
# random_state=42 - fixes the random shuffle so your split matches mine (reproducibility)
# this is the SAME train_test_split idea from your sklearn projects,
# just surprise's own version that understands user-item rating data

print(f"Training ratings: {trainset.n_ratings:,}")
print(f"Test ratings:     {len(testset):,}")
# does this split look right? 80/20 of 100,836...

Training ratings: 80,668
Test ratings:     20,168


In [57]:
# Day 3 Part B, Step 4 - train the SVD matrix factorization model

from surprise import SVD

model = SVD(n_factors=100, random_state=42)
# SVD - Singular Value Decomposition, the factorization algorithm
#       (this is the one that won the Netflix Prize)
# n_factors=100 - how many HIDDEN dimensions to discover.
#                 each movie & each user gets described by 100 learned numbers.
#                 more factors = more nuance, but slower and risks overfitting
# random_state=42 - reproducible results

model.fit(trainset)
# .fit() - learns the user & movie factor profiles from the training ratings
#          SAME .fit() verb you've used on every sklearn model
#          this runs gradient descent under the hood, may take a few seconds

print("Model trained.")

Model trained.


In [58]:
# Day 3 Part B, Step 5 - measure prediction error on the held-out test set

from surprise import accuracy

predictions = model.test(testset)
# .test() - runs the trained model on the test ratings,
#           producing a predicted rating for each (user, movie) pair
#           it predicts WITHOUT seeing the true rating, then we compare

rmse = accuracy.rmse(predictions)
# RMSE - Root Mean Squared Error: average size of the prediction error,
#        in the SAME units as the ratings (stars).
#        squares the errors (punishes big misses harder), averages, square-roots back.
#        RMSE of 0.9 means predictions are off by ~0.9 stars on average.

mae = accuracy.mae(predictions)
# MAE - Mean Absolute Error: plain average of how far off, no squaring.
#       gentler on big misses. usually a bit lower than RMSE.

RMSE: 0.8807
MAE:  0.6766


In [60]:
# Day 3 Part B, Step 6 - recommend top movies for a specific USER

def recommend_for_user(user_id, n=10):
    # predict this user's rating for every movie, return the top n unseen ones

    # movies this user has ALREADY rated - we don't recommend those back
    rated = ratings[ratings['userId'] == user_id]['movieId'].tolist()
    # filter ratings to this user, grab their movieIds, as a plain list

    # every movieId in the dataset
    all_movies = ratings['movieId'].unique()

    #predict a rating for every movie the user HASN'T rated
    preds = []
    for mid in all_movies:
        if mid not in rated: # skip already-seen movies
            pred = model.predict(user_id, mid) # predict this user's rating for this movie
            preds.append((mid, pred.est)) # pred.est - the estimated rating
            # we collect (movieId, predicted_rating) pairs
    
    # sort by predicted rating, highest first, take top n
    preds.sort(key=lambda x: x[1], reverse=True)
    top = preds[:n]

    # translate movieIds → titles for display
    top_ids = [mid for mid, est in top]
    result = movies[movies['movieId'].isin(top_ids)][['title']].copy()
    result['title'] = result['title'].apply(fix_title)   # reuse the Day 2 beautifier
    return result

# recommend for user 1
recommend_for_user(1)

,title
474,Blade Runner (1982)
602,Dr. Strangelove or: How I Learned to Stop Worr...
690,North by Northwest (1959)
694,Casablanca (1942)
896,One Flew Over the Cuckoo's Nest (1975)
924,A Grand Day Out with Wallace and Gromit (1989)
1494,Seven Samurai (Shichinin no samurai) (1954)
4529,Lost in Translation (2003)
4800,The Lord of the Rings: The Return of the King ...
6315,The Departed (2006)


In [61]:
# Day 3 Part B, Step 7 - FIXED recommender: preserves predicted-rating order

def recommend_for_user(user_id, n=10):
    rated = set(ratings[ratings['userId'] == user_id]['movieId'])
    # set() - faster lookups than a list when we check "not in rated" many times

    all_movies = ratings['movieId'].unique()

    # predict for every unseen movie, collect (movieId, predicted_rating) pairs
    preds = [(mid, model.predict(user_id, mid).est)
             for mid in all_movies if mid not in rated]
    # list comprehension version of the loop - same logic, more compact

    preds.sort(key=lambda x: x[1], reverse=True)   # sort best-first
    top = preds[:n]                                 # take top n, STILL in order

    # THE FIX: build the result by walking `top` in order, not via isin()
    title_lookup = movies.set_index('movieId')['title']
    # set_index('movieId') - makes movieId the lookup key, so title_lookup[mid] → title

    rows = [{'title': fix_title(title_lookup[mid]),    # reuse Day 2 beautifier
             'predicted_rating': round(est, 2)}        # show the score too
            for mid, est in top]                       # walk top IN ORDER

    return pd.DataFrame(rows)

recommend_for_user(1)

,title,predicted_rating
0,The Departed (2006),5.0
1,North by Northwest (1959),5.0
2,Casablanca (1942),5.0
3,Seven Samurai (Shichinin no samurai) (1954),5.0
4,Dr. Strangelove or: How I Learned to Stop Worr...,5.0
5,The Lord of the Rings: The Return of the King ...,5.0
6,Blade Runner (1982),5.0
7,One Flew Over the Cuckoo's Nest (1975),5.0
8,A Grand Day Out with Wallace and Gromit (1989),5.0
9,Lost in Translation (2003),5.0
